In [6]:
#Importing files
import pandas as pd
import requests
import time
#from wordcloud import WordCloud ill fix this later
from collections import Counter
import matplotlib.pyplot as plt
import re


In [7]:
#loading the data and filtering this (in terms of realistic-ness)
df = pd.read_csv('evaluation_results3.csv')

df['Realistic?'] = df['Realistic?'].fillna('Yes')
df = df[df['Realistic?'].str.contains('Yes', case=False, na=False)]

print(f"Loaded {len(df)} realistic requests.")

Loaded 1412 realistic requests.


In [10]:
# extracting levels

def extract_levels(value):
    if pd.isna(value) or value == 'Original Request':
        return None, None
    if ' + ' in value:
        parts = value.split(' + ')
        if len(parts) == 2:
            seniority = parts[0].strip().title()
            raw_hastiness = parts[1].strip().title().split('(')[0].strip()
            
            if 'Very Hasty' in raw_hastiness:
                hastiness = 'Very Hasty'
            elif 'Hasty' in raw_hastiness:
                hastiness = 'Hasty'
            elif 'Neutral' in raw_hastiness:
                hastiness = 'Neutral'
            elif 'Formal' in raw_hastiness:
                hastiness = 'Formal'
            elif 'Very Formal' in raw_hastiness:
                hastiness = 'Very Formal'
            else:
                hastiness = raw_hastiness
            return seniority, hastiness
    return None, None

In [11]:
df['Seniority'] = df['Variation Value'].apply(lambda x: extract_levels(x)[0])
df['Hastiness'] = df['Variation Value'].apply(lambda x: extract_levels(x)[1])
df = df.dropna(subset=['Hastiness'])

print(f"Rows with hastiness: {len(df)}")


Rows with hastiness: 1308


In [13]:
#cleaning text (removing stopwords)
STOPWORDS = set([
    'to', 'for', 'the', 'of', 'and', 'in', 'on', 'at', 'with', 'by', 'from', 
    'as', 'is', 'was', 'we', 'our', 'us', 'i', 'you', 'this', 'that', 
    'these', 'those', 'a', 'an', 'data', 'access', 'request', 'will', 'can',
    'use', 'used', 'using', 'analysis', 'analyze', 'help', 'make', 'need',
    'would', 'could', 'should', 'may', 'might', 'must', 'shall', 'also',
    'well', 'better', 'good', 'great', 'really', 'very', 'even'
])

def clean_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return [w for w in words if w not in STOPWORDS and len(w) > 2]

df['Clean_Words'] = df['Purpose'].apply(clean_text)


In [14]:
def get_ngrams_freq(word):
    """Query NGRAMS API for relative frequency of a word in English."""
    try:
        url = f"https://api.ngrams.dev/eng/search?query={word}&flags=cr"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('ngrams'):
                return data['ngrams'][0].get('relTotalMatchCount', 0)
    except Exception as e:
        pass
    return 0
    

In [17]:
import pandas as pd
import requests
import time
from collections import Counter
import matplotlib.pyplot as plt
import re

# --- 1. Load Data and Filter ---
df = pd.read_csv('evaluation_results3.csv')
df['Realistic?'] = df['Realistic?'].fillna('Yes')
df = df[df['Realistic?'].str.contains('Yes', case=False, na=False)]

# --- 2. Extract Levels ---
def extract_levels(value):
    if pd.isna(value) or value == 'Original Request':
        return None, None
    if ' + ' in value:
        parts = value.split(' + ')
        if len(parts) == 2:
            seniority = parts[0].strip().title()
            raw_hastiness = parts[1].strip().title().split('(')[0].strip()
            if 'Very Hasty' in raw_hastiness:
                hastiness = 'Very Hasty'
            elif 'Hasty' in raw_hastiness:
                hastiness = 'Hasty'
            elif 'Neutral' in raw_hastiness:
                hastiness = 'Neutral'
            elif 'Formal' in raw_hastiness:
                hastiness = 'Formal'
            elif 'Very Formal' in raw_hastiness:
                hastiness = 'Very Formal'
            else:
                hastiness = raw_hastiness
            return seniority, hastiness
    return None, None

df['Seniority'] = df['Variation Value'].apply(lambda x: extract_levels(x)[0])
df['Hastiness'] = df['Variation Value'].apply(lambda x: extract_levels(x)[1])
df = df.dropna(subset=['Hastiness'])

# --- 3. Clean Text ---
STOPWORDS = set([
    'to', 'for', 'the', 'of', 'and', 'in', 'on', 'at', 'with', 'by', 'from', 
    'as', 'is', 'was', 'we', 'our', 'us', 'i', 'you', 'this', 'that', 
    'these', 'those', 'a', 'an', 'data', 'access', 'request', 'will', 'can',
    'use', 'used', 'using', 'analysis', 'analyze', 'help', 'make', 'need',
    'would', 'could', 'should', 'may', 'might', 'must', 'shall'
])

def clean_text(text):
    if not isinstance(text, str):
        return []
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    return [w for w in words if w not in STOPWORDS and len(w) > 2]

df['Clean_Words'] = df['Purpose'].apply(clean_text)

# --- 4. Get NGRAMS Baseline ---
def get_ngrams_freq(word):
    try:
        url = f"https://api.ngrams.dev/eng/search?query={word}&flags=cr"
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get('ngrams'):
                return data['ngrams'][0].get('relTotalMatchCount', 0)
    except Exception:
        pass
    return 0

all_words = [w for words in df['Clean_Words'] for w in words]
unique_words = list(set(all_words))

print(f"Fetching baseline frequencies for {len(unique_words)} words...")
ngram_baseline = {}
for i, word in enumerate(unique_words):
    if i % 20 == 0:
        print(f"Progress: {i}/{len(unique_words)}")
    ngram_baseline[word] = get_ngrams_freq(word)
    time.sleep(0.15)

# --- 5. Calculate Lift Scores ---
def get_lift_words(df, group_value, baseline, top_n=15):
    group_df = df[df['Hastiness'] == group_value]
    group_words = [w for words in group_df['Clean_Words'] for w in words]
    group_counter = Counter(group_words)
    total_group_words = len(group_words)
    
    scores = {}
    for word, count in group_counter.items():
        actual_freq = count / total_group_words if total_group_words > 0 else 0
        expected_freq = baseline.get(word, 1e-12)
        if expected_freq > 0 and actual_freq > 0:
            scores[word] = actual_freq / expected_freq
    
    sorted_words = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:top_n]
    return sorted_words

hasty_lift = get_lift_words(df, 'Very Hasty', ngram_baseline)
formal_lift = get_lift_words(df, 'Very Formal', ngram_baseline)

# --- 6. Plot Bar Charts (No wordcloud library needed!) ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

if hasty_lift:
    words, scores = zip(*hasty_lift)
    axes[0].barh(words, scores, color='red')
    axes[0].set_xlabel('Lift Score (Actual / Expected)')
    axes[0].set_title('Words Overused in "Very Hasty" Requests')
    axes[0].invert_yaxis()

if formal_lift:
    words, scores = zip(*formal_lift)
    axes[1].barh(words, scores, color='blue')
    axes[1].set_xlabel('Lift Score (Actual / Expected)')
    axes[1].set_title('Words Overused in "Very Formal" Requests')
    axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# --- 7. Print Results ---
print("\nTop 10 Overused Words in Hasty Requests:")
for word, score in hasty_lift[:10]:
    print(f"  {word}: {score:.1f}x")

print("\nTop 10 Overused Words in Formal Requests:")
for word, score in formal_lift[:10]:
    print(f"  {word}: {score:.1f}x")
    

Fetching baseline frequencies for 2378 words...
Progress: 0/2378
Progress: 20/2378
Progress: 40/2378
Progress: 60/2378
Progress: 80/2378
Progress: 100/2378
Progress: 120/2378
Progress: 140/2378
Progress: 160/2378
Progress: 180/2378
Progress: 200/2378
Progress: 220/2378


KeyboardInterrupt: 